# Focusing NLSE: a traveling bright soliton in JAX

Solve $i\psi_t=-\psi_{xx}-g|\psi|^2\psi$, with $g=2$.
Focusing nonlinearity balances dispersion, so a properly matched sech pulse
retains its shape. This is a separate nonlinear example; the linear
Schrödinger notebook still demonstrates a dispersing Gaussian.

Install `python -m pip install -e './jax[notebook]'` from the repository root.
Use an x64 kernel. On the affected OpenMP BLAS installation, set
`OMP_NUM_THREADS=1` before Python starts or use the corrected-BLAS launcher.
First calls include compilation.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import bspf_jax as bspf

## Model, reference, and boundaries

For $g=2$, the infinite-line soliton is
$$\psi(x,t)=\operatorname{sech}(x-x_0-2kt)
\exp\{i[kx+(1-k^2)t]\}.$$
Choose $x_0=-3$ and $k=0.5$: the center travels from -3 to 1 over $0\le t\le4$,
with constant amplitude and width. The finite domain is $[-28,28]$, with
natural zero-flux weak boundaries. The reference is an infinite-line solution,
not an exact finite-box solution: its boundary tails stay below $3\times10^{-11}$.
The pulse never reaches a wall during this test.

BSPF mass, stiffness, and the **cubic projection** use resolved quadrature.
`integrate_nlse` applies a fourth-order interaction-picture RK method: exact
linear modal phases plus projected nonlinear stages. It is not exactly
norm/energy preserving; step refinement and both invariants are checked.

In [ ]:
coupling, x0, wave_number = 2., -3., 0.5
times = jnp.linspace(0., 4., 41)

def reference(x, t):
    center = x0+2*wave_number*t
    return jnp.exp(1j*(wave_number*x+(1-wave_number**2)*t))/jnp.cosh(x-center)

def spatial_plan(n):
    x = jnp.linspace(-28., 28., n)
    plan = bspf.plan_1d(x, degree=7, n_basis=48, boundary_points=9)
    return x, bspf.galerkin_1d(plan, quadrature_order=8)

def evolve(weak, x, substeps):
    return jax.jit(lambda: bspf.integrate_nlse(
        weak, reference(x, 0.), times, coupling=coupling, substeps=substeps))()

x, weak = spatial_plan(513)
solution = evolve(weak, x, 40)  # dt = 0.0025
larger_step = evolve(weak, x, 20)
coarse_x, coarse_weak = spatial_plan(257)
coarse = evolve(coarse_weak, coarse_x, 40)
linear = jax.jit(bspf.integrate_schrodinger)(weak.mass, weak.stiffness, reference(x, 0.), times)

## Accuracy, shape, and conservation

Compare the full complex field (including phase), not only density.
The Hamiltonian is $H=\psi^*K\psi-(g/2)\int|\psi|^4dx$; evaluate its nonlinear
term at the same quadrature points used by the integrator. The width diagnostic
uses the second moment of the sampled density. For this soliton its exact
standard deviation is $\pi/\sqrt{12}$.

The fine grid, coarse grid, and doubled time step are separate runs.
No normalization, phase alignment, or filtering is applied during evolution.

In [ ]:
exact = reference(x[None, :], times[:, None])
field_error = jnp.max(jnp.abs(solution-exact), axis=1)
coarse_error = jnp.max(jnp.abs(coarse-reference(coarse_x[None, :], times[:, None])))
step_error = jnp.max(jnp.abs(larger_step-exact))
time_difference = jnp.max(jnp.abs(solution-larger_step))
norm = jnp.real(jnp.einsum("ti,ij,tj->t", solution.conj(), weak.mass, solution))
quadrature_field = solution@weak.values.T
energy = (jnp.real(jnp.einsum("ti,ij,tj->t", solution.conj(), weak.stiffness, solution))
          -coupling/2*(jnp.abs(quadrature_field)**4)@weak.quadrature_weights)
norm_drift = jnp.abs(norm/norm[0]-1)
energy_drift = jnp.abs((energy-energy[0])/jnp.abs(energy[0]))
density = jnp.abs(solution)**2
center = x0+2*wave_number*times
width = jnp.sqrt(jnp.trapezoid((x[None, :]-center[:, None])**2*density, x, axis=1)
                 /jnp.trapezoid(density, x, axis=1))
width_error = jnp.max(jnp.abs(width-jnp.pi/jnp.sqrt(12)))
boundary_tail = jnp.max(jnp.abs(exact[:, jnp.array([0, -1])] ))
print(f"maximum complex field error: {field_error.max():.3e}; coarse grid: {coarse_error:.3e}")
print(f"doubled-step error: {step_error:.3e}; step difference: {time_difference:.3e}")
print(f"relative norm drift: {norm_drift.max():.3e}; relative energy drift: {energy_drift.max():.3e}")
print(f"width error: {width_error:.3e}; reference boundary tail: {boundary_tail:.3e}")
assert jnp.all(jnp.isfinite(solution))
assert field_error.max() < 5e-9
assert field_error.max() < coarse_error/5
assert field_error.max() < step_error/5
assert norm_drift.max() < 1e-10
assert energy_drift.max() < 1e-8
assert width_error < 1e-8
assert boundary_tail < 3e-11

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
im = axes[0, 0].pcolormesh(x, times, density, shading="auto", cmap="magma")
axes[0, 0].plot(center, times, "c--", linewidth=1, label="Exact center")
axes[0, 0].set(xlim=(-10, 10), xlabel="x", ylabel="t", title="Traveling soliton density")
axes[0, 0].legend()
fig.colorbar(im, ax=axes[0, 0])
axes[0, 1].plot(x, density[-1], label="NLSE")
axes[0, 1].plot(x, jnp.abs(exact[-1])**2, "k--", label="Exact soliton")
axes[0, 1].plot(x, jnp.abs(linear[-1])**2, label="Linear evolution, same initial pulse")
axes[0, 1].set(xlim=(-12, 14), xlabel="x", ylabel="Density", title="Nonlinearity balances dispersion at t=4")
axes[0, 1].legend()
axes[1, 0].semilogy(times[1:], field_error[1:], label="dt=0.0025")
axes[1, 0].semilogy(times[1:], jnp.max(jnp.abs(larger_step-exact), axis=1)[1:], label="dt=0.005")
axes[1, 0].set(xlabel="t", ylabel="Max complex field error", title="Time refinement")
axes[1, 0].legend()
axes[1, 1].semilogy(times, jnp.maximum(norm_drift, 1e-17), label="Norm")
axes[1, 1].semilogy(times, jnp.maximum(energy_drift, 1e-17), label="Hamiltonian")
axes[1, 1].set(xlabel="t", ylabel="Relative drift", title="Conservation diagnostics")
axes[1, 1].legend()
plt.show()